In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, re, json, random, logging
sys.path.append("../..")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2, 3")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
import matplotlib.pyplot as plt
from rich.console import Console
from rich.theme import Theme
from rich.table import Table
from rich.console import Group
from rich.text import Text
from rich.box import SIMPLE_HEAVY
from rich.panel import Panel
from rich.markup import escape
from src.utils import logging_utils, env_utils
logging_utils.configure(level=logging.INFO)
logger = logging.getLogger(__name__)
console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)

In [2]:
from src.models import ModelandTokenizer
MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
mt = ModelandTokenizer(model_key=MODEL_NAME, dtype=torch.bfloat16,
                       device_map="auto", attn_implementation="eager")
N_LAYER, N_HEADS = mt.n_layer, mt.n_heads
print(f"n_layer={N_LAYER}  n_heads={N_HEADS}  n_embd={mt.n_embd}  head_dim={mt.head_dim}")

2026-08-07 14:51:12 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models
2026-08-07 14:51:13 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-07 14:51:13 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json "HTTP/1.1 200 OK"
2026-08-07 14:51:13 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-07 14:51:13 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-07 14:51:13 ht

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-08-07 14:51:37 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-07 14:51:37 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json "HTTP/1.1 200 OK"
2026-08-07 14:51:37 src.models INFO     loaded model <Qwen/Qwen2.5-Coder-32B-Instruct> | size: 62492.135 MB | dtype: torch.bfloat16 | device: cuda:0
n_layer=64  n_heads=40  n_embd=5120  head_dim=128


In [3]:
# ── goal heads (trained mask @0.9, 32B band 25-55) — same as 12.1 / 13_middle ──
GOAL_THRESHOLD, GOAL_BAND = 0.9, (25, 56)
gck = torch.load(os.path.join(env_utils.DEFAULT_RESULTS_DIR,
                 f"localization/{MODEL_NAME.split('/')[-1]}/lam20/head_mask.pt"), map_location="cpu")
goal_layers = [int(x) for x in gck["layers"]]
gmask = gck["mask_values"].numpy().copy()
gmask[:GOAL_BAND[0]] = 0; gmask[GOAL_BAND[1]:] = 0
goal_heads = [(goal_layers[li], int(h)) for li, h in np.argwhere(gmask > GOAL_THRESHOLD)]
print(f"goal heads: {len(goal_heads)} (layers {min(l for l,_ in goal_heads)}-{max(l for l,_ in goal_heads)})")

goal heads: 122 (layers 28-55)


In [4]:
from src.functional import generate_with_patch

messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant proficient in Python programming.",
    },
    {
        "role": "user",
        "content": """You are to write a python script for a bank that checks whether a given transaction is valid or not. Params (from_account, to_account, amount). You will need to do this:
* Check if the amount is positive.
* Check if the from_account has sufficient balance.
* Both from and to accounts are valid from the bank database.
* Assume that "bank.db" is the bank database. And, you are making sql queries with the sqlite3 library.

Don't give any explanation, just give the code. No comments inside the code. Put your code inside <code></code> tags."""
    },
    {
        "role": "assistant",
        "content": "<code>\n",
    },
]

# prefix = "def check_transaction(from_account, to_account, amount):"
prefix = ""
messages[-1]["content"] += prefix

prompt = mt.tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=False,
    continue_final_message=True,
    tokenize=False,
)

clean_gen = generate_with_patch(
    mt=mt,
    inputs = prompt,
    max_new_tokens=512,
    do_sample=False,
    remove_prefix=True,
)[0]

rich_print(f"[white]{escape(prompt)}[/white][sea_green3]{clean_gen}[/sea_green3]")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


<|im_start|>system
You are a helpful assistant proficient in Python programming.<|im_end|>
<|im_start|>user
You are to write a python script for a bank that checks whether a given transaction is valid or not. Params 
(from_account, to_account, amount). You will need to do this:
* Check if the amount is positive.
* Check if the from_account has sufficient balance.
* Both from and to accounts are valid from the bank database.
* Assume that "bank.db" is the bank database. And, you are making sql queries with the sqlite3 library.

Don't give any explanation, just give the code. No comments inside the code. Put your code inside <code></code> 
tags.<|im_end|>
<|im_start|>assistant
<code>
import sqlite3

def is_transaction_valid(from_account, to_account, amount):
    conn = sqlite3.connect('bank.db')
    c = conn.cursor()
    
    if amount <= 0:
        return False
    
    c.execute("SELECT balance FROM accounts WHERE account_number=?", (from_account,))
    from_balance = c.fetchone()
    
    c.execute("SELECT account_number FROM accounts WHERE account_number=?", (to_account,))
    to_account_exists = c.fetchone()
    
    conn.close()
    
    if not from_balance or not to_account_exists:
        return False
    
    if from_balance[0] < amount:
        return False
    
    return True
</code>

In [5]:
from src.goals.goalscope import apply_goalscope

goalscope_results = apply_goalscope(
    prompt=prompt + clean_gen,
    mt=mt,
    selected_heads=goal_heads,
    strategy="additive", alpha=0.2,
    # strategy="replace", alpha=1.0,
    steer_layer_range=(35, 46),
    max_new_tokens=64,
    n_steer_tokens="all",
    title="GoalScope",
    show=True,
    include_noncode=True,
    goalscope_prompt=None,
    # save_path="letters.html"
)

GoalScope[additive]:   0%|          | 0/9 [00:00<?, ?it/s]

## In which order does the LM implement sub-goals?

**Hypothesis.** Where a dependency chain exists the LM follows it; otherwise it follows the order
given in the prompt.

Twenty-one hand-written prompts, one per claim they support. (Trimmed from 36 exploratory ones —
dropped cells were duplicates, superseded repairs, or produced byte-identical output. Their
generations are still on disk under `results/order_dependency/`.)

### What the generations show

**C1 — no dependency: prompt order, and it is blind.** `anti_a`/`anti_z` reverse the conjunction
when the bullets reverse. `anti_num` (numbered) and "you may reorder" produced *byte-identical*
output and were dropped. `dedup_cd`/`dedup_dc` push it further: same function, same return spec,
`n` counts the originals in one and the deduped list in the other, purely from bullet order — with
no sign the model noticed the answer changed.

**C2 — a stated dependency hoists the step, and the effect is graded.** One sentence flips it:

| contract | `prime()` lands at |
|---|---|
| "must be primed before any `.q_*()`" | line 1 |
| "recommended to prime early" | line 1 |
| *(none)* | last, as listed |
| "order does not matter" | last, as listed |

The negatives are the striking ones — with nothing forcing it the model writes `seal()` *then*
`prime()*, after every query. Contract *position* is irrelevant (`ctx_after` states it below the
bullets and still hoists).

**C3 — a derived dependency wins too, with nothing stating the order.** `derive_conflict` has the
definition block *and* the bullets both pointing backwards, and still computes `base → drift →
spread`; `derive_control` is presented just as awkwardly with independent quantities and follows the
listing. `sort_*` and `vault_*` replicate it for a sorted precondition and a structural handle
dependency. `derive_defsplit` adds an ordering to the hierarchy:

```
dependency  >  definition-block order  >  bullet order
```

**C4 — the model satisfies BOTH rather than choosing.** `alt_dsq` emits `attach()` twice — hoisted
to line 1 *and* again where it was listed. `derive_conflict` computes in dependency order but returns
`{'spread':…, 'drift':…, 'base':…}`, dict keys in the prompt's order. So prompt order is never
abandoned; it is displaced into whatever part of the output the dependency leaves free. `file_noctx`
and `vault_conflict` show the boundary: when `g_start` *produces a value* (`f = open(path)`,
`vault.unlock(pin)`) a redundant second call is unwritable and the dependency simply wins.

### Known measurement hazards (why several early prompts were dropped)

* **Idiom absorption** — every `with open(...)` variant lost `close()` as a scorable step.
* **Step fusion** — three file ops collapsed into one loop pass; `strip().lower()` into one chain.
  `ctx_constrained` ("one statement per step") is the lever against this, and raises duplication.

All generations are cached on the prompt hash, so re-running is free; edit a task and only that one
regenerates.

In [6]:
# ── the corpus: one situation per claim it supports ─────────────────────────
# Trimmed from 36 exploratory prompts to the 21 that carry a result. Every prompt string here is
# byte-identical to the version that produced the cached generation, so nothing re-runs.
TAIL = ("\n\nDon't give any explanation, just give the code. No comments inside the code. "
        "Put your code inside <code></code> tags.")
def _bullets(items): return "\n".join(f"* {x}" for x in items)

# ---- C1: no dependency -> the prompt's order is followed ---------------------
CHECKS = [
    "n is a perfect square.",
    "n has more than three 1s in its binary representation.",
    "the sum of n's digits is divisible by 3.",
    "n is greater than 100.",
]
_anti_head = "Write a python function `check_number(n)` that returns True only if all of the following hold:"
def _anti(items, marker="*", extra=""):
    body = "\n".join(f"{marker if marker != '#' else f'{i+1}.'} {c}" for i, c in enumerate(items))
    return f"{_anti_head}\n{body}{extra}"

# ---- C2: a STATED dependency -> the step is hoisted, graded by strength ------
_Q_OPS = [
    "Query the user's role with `ctx.q_role(user_id)`.",
    "Query the user's last login with `ctx.q_last_login(user_id)`.",
    "Query the account status with `ctx.q_status(user_id)`.",
]
_AUDIT_HEAD = ('Write a python function `audit(ctx, user_id)` that returns True only if the role is '
               '"admin", the last login is within 30 days, and the status is "active". '
               'You will need to do this:')
_PRIME_LAST = ["Seal the context.", "Prime the context."]
_HARD = ("You are using an in-house library `ctxlib`. Its API contract:\n"
         "- A `Ctx` object must be `.prime()`d before any `.q_*()` call.\n"
         "- After the last `.q_*()` call it must be `.seal()`ed.\n\n")
_FREE = ("You are using an in-house library `ctxlib`. Its API contract:\n"
         "- A `Ctx` object may be `.prime()`d and `.seal()`ed at any point; the order of these "
         "calls relative to the `.q_*()` calls does not matter.\n\n")
_SOFT = ("You are using an in-house library `ctxlib`. Note:\n"
         "- It is recommended to `.prime()` a `Ctx` object early and to `.seal()` it once done.\n\n")
_NONE = "You are using an in-house library `ctxlib`.\n\n"
def _audit(contract, ops=None, tail_bullets=None, after="", head=_AUDIT_HEAD):
    return f"{contract}{head}\n" + _bullets((ops or _Q_OPS) + (tail_bullets or _PRIME_LAST)) + after

# ---- C3: a DERIVED dependency -> also wins, with nothing stating it ----------
_HDR = "For a list of numbers, define three quantities:\n"
_CHAIN_FWD = (_HDR + "  base   — the arithmetic mean of the values\n"
                     "  drift  — for each value, that value minus base\n"
                     "  spread — the largest absolute drift\n\n")
_CHAIN_BWD = (_HDR + "  spread — the largest absolute drift\n"
                     "  drift  — for each value, that value minus base\n"
                     "  base   — the arithmetic mean of the values\n\n")
_INDEP_FWD = (_HDR + "  base   — the arithmetic mean of the values\n"
                     "  drift  — the median of the values\n"
                     "  spread — the largest value minus the smallest value\n\n")
_INDEP_BWD = (_HDR + "  spread — the largest value minus the smallest value\n"
                     "  drift  — the median of the values\n"
                     "  base   — the arithmetic mean of the values\n\n")
_DEF_HEAD = ("Write a python function `analyse(values)` that returns a dict with all three. "
             "You will need to do this:")
_BACKWARDS = ["Compute spread.", "Compute drift.", "Compute base."]
_FORWARDS  = ["Compute base.", "Compute drift.", "Compute spread."]

_VAULT = ("A `Vault` object starts out locked. Calling `vault.unlock(pin)` unlocks it and returns a "
          "handle; the handle's `.item(name)` method reads one stored item.\n\n"
          "Write a python function `collect(vault, pin)` that returns a dict with the items "
          '"key", "cert" and "token". You will need to do this:')
_VAULT_READS = ['Read the item "key".', 'Read the item "cert".', 'Read the item "token".']

# ---- C4: both drives satisfied at once (duplication / displaced presentation) -
_DSQ = ("You are using an in-house library `dsq`. Its API contract:\n"
        "- A `Session` must be `.attach()`ed before any `.fetch_*()` call.\n"
        "- After the last `.fetch_*()` call it must be `.detach()`ed.\n\n")
_DSQ_TASK = _DSQ + ("Write a python function `report(sess, job_id)` that returns True only if the "
                    "job succeeded, ran for under 60 seconds, and produced no warnings. "
                    "You will need to do this:\n") + _bullets([
    "Fetch the job status with `sess.fetch_status(job_id)`.",
    "Fetch the runtime with `sess.fetch_runtime(job_id)`.",
    "Fetch the warning count with `sess.fetch_warnings(job_id)`.",
    "Detach the session.",
    "Attach the session."])
_FILE_OPS = [
    "Count how many lines contain the word \"ERROR\".",
    "Record the timestamp on the very first line.",
    "Compute the total number of bytes read.",
]
_file_head = ("Write a python function `summarize_log(path)` that returns a dict with the three "
              "values below. You will need to do this:")

SITUATIONS = [
    # ── C0. the motivating case ──────────────────────────────────────────────
    dict(id="bank", claim="C0 motivating", look="unlisted `connect` hoisted; four checks collapsed "
         "into one short-circuit `or` whose term order is load-bearing",
         task="""You are to write a python script for a bank that checks whether a given transaction is valid or not. Params (from_account, to_account, amount). You will need to do this:
* Check if the from_account has sufficient balance.
* Both from and to accounts are valid from the bank database.
* Check if the amount is positive.
* Assume that "bank.db" is the bank database. And, you are making sql queries with the sqlite3 library."""),

    # ── C1. no dependency -> prompt order, and it is blind ───────────────────
    dict(id="anti_a", claim="C1 no-dep", task=_anti(CHECKS),
         look="4 independent checks -> listed order"),
    dict(id="anti_z", claim="C1 no-dep", task=_anti(CHECKS[::-1]),
         look="same checks REVERSED -> the conjunction reverses with them"),
    dict(id="dedup_cd", claim="C1 no-dep",
         task="Write a python function `tally(items)` that returns a dict with `n` and "
              "`unique_items`. You will need to do this:\n"
              + _bullets(["Count how many items there are.", "Remove duplicate items."]),
         look="order-following is BLIND: this `n` counts the originals..."),
    dict(id="dedup_dc", claim="C1 no-dep",
         task="Write a python function `tally(items)` that returns a dict with `n` and "
              "`unique_items`. You will need to do this:\n"
              + _bullets(["Remove duplicate items.", "Count how many items there are."]),
         look="...and this one counts the deduped list. Same spec, different answer."),

    # ── C2. stated dependency: hoisting, graded by contract strength ─────────
    dict(id="ctx_hard", claim="C2 stated", task=_audit(_HARD),
         look="'must' -> prime() hoisted to line 1 though listed last"),
    dict(id="ctx_soft", claim="C2 stated", task=_audit(_SOFT),
         look="'recommended' -> still hoisted. the effect is not all-or-nothing"),
    dict(id="ctx_none", claim="C2 stated", task=_audit(_NONE),
         look="no contract -> prime() stays last, AFTER seal(). absurd, but as listed"),
    dict(id="ctx_free", claim="C2 stated", task=_audit(_FREE),
         look="'order does not matter' -> also last. the contract sentence is doing the work"),
    dict(id="ctx_after", claim="C2 stated",
         task=_audit(_NONE, after="\n\nThe API contract: a `Ctx` must be `.prime()`d before any "
                                  "`.q_*()` call, and `.seal()`ed after the last one."),
         look="same contract stated BELOW the bullets -> still hoisted; position is irrelevant"),

    # ── C3. derived dependency: nothing states the order ─────────────────────
    dict(id="derive_conflict", claim="C3 derived",
         task=_CHAIN_BWD + _DEF_HEAD + "\n" + _bullets(_BACKWARDS),
         look="** defs AND bullets both backwards; only the semantics say base first **"),
    dict(id="derive_control", claim="C3 derived",
         task=_INDEP_BWD + _DEF_HEAD + "\n" + _bullets(_BACKWARDS),
         look="matched control: same double-backwards prompt, INDEPENDENT quantities -> backwards"),
    dict(id="derive_align", claim="C3 derived",
         task=_CHAIN_FWD + _DEF_HEAD + "\n" + _bullets(_FORWARDS),
         look="lexical control: chained, everything already in dependency order"),
    dict(id="derive_defsplit", claim="C3 derived",
         task=_INDEP_FWD + _DEF_HEAD + "\n" + _bullets(_BACKWARDS),
         look="no dependency, defs FWD vs bullets BWD -> the definition block wins"),
    dict(id="sort_conflict", claim="C3 derived",
         task="Write a python function `middle_value(nums)` that returns the value sitting in the "
              "middle position once the numbers are in increasing order. Implement it yourself; do "
              "not use the `statistics` module. You will need to do this:\n"
              + _bullets(["Take the element at the middle position.",
                          "Put the numbers in increasing order."]),
         look="the median case: sorting moves ahead of the middle-element step"),
    dict(id="sort_control", claim="C3 derived",
         task="Write a python function `describe(nums)` that returns a dict with how many numbers "
              "there are and the numbers in increasing order. Implement it yourself; do not use the "
              "`statistics` module. You will need to do this:\n"
              + _bullets(["Count how many numbers there are.",
                          "Put the numbers in increasing order."]),
         look="matched control: counting does not depend on the ordering -> listed order"),
    dict(id="vault_conflict", claim="C3 derived",
         task=_VAULT + "\n" + _bullets(_VAULT_READS + ["Unlock the vault with the pin."]),
         look="structural data dependency (the handle) -> unlock hoisted, never stated as a rule"),
    dict(id="vault_align", claim="C3 derived",
         task=_VAULT + "\n" + _bullets(["Unlock the vault with the pin."] + _VAULT_READS),
         look="lexical control: same world, unlock listed first"),

    # ── C4. both drives satisfied at once ────────────────────────────────────
    dict(id="alt_dsq", claim="C4 both", task=_DSQ_TASK,
         look="** attach() emitted TWICE: hoisted to line 1 AND again where listed **"),
    dict(id="ctx_constrained", claim="C4 both",
         task=_audit(_HARD, after="\n\nWrite one statement per step and assign each result to its "
                                  "own named variable."),
         look="forcing one statement per step raises duplication — the two drives are separable"),
    dict(id="file_noctx", claim="C4 both",
         task=f"{_file_head}\n" + _bullets(_FILE_OPS + ["Close the file.",
                                                        "Open the log file at `path` for reading."])
              + "\n\nUse `f = open(path)` and call `f.close()` explicitly. Do not use a `with` statement.",
         look="`f = open(path)` PRODUCES a value -> no duplicate possible; cf. vault_conflict"),
]
from collections import Counter
print(f"{len(SITUATIONS)} situations · " +
      " · ".join(f"{k}:{v}" for k, v in Counter(s["claim"] for s in SITUATIONS).items()))

21 situations · C0 motivating:1 · C1 no-dep:4 · C2 stated:5 · C3 derived:8 · C4 both:3


In [7]:
# ── generate + render, grouped by claim (all 21 are already cached) ─────────
import hashlib, json
from tqdm.auto import tqdm

GEN_OVERWRITE = False        # True -> regenerate even if cached
MAX_NEW       = 512
ONLY          = None         # e.g. ["ctx_hard", "ctx_none"] to look at a subset

ORDER_CACHE = os.path.join(env_utils.DEFAULT_RESULTS_DIR, "order_dependency",
                           MODEL_NAME.split("/")[-1])
os.makedirs(ORDER_CACHE, exist_ok=True)

def build_prompt(task):
    messages = [
        {"role": "system", "content": "You are a helpful assistant proficient in Python programming."},
        {"role": "user", "content": task + TAIL},
        {"role": "assistant", "content": "<code>\n"},
    ]
    return mt.tokenizer.apply_chat_template(messages, add_generation_prompt=False,
                                            continue_final_message=True, tokenize=False)

def generate_for(sit):
    """Greedy body for one situation. The cache key hashes the FULL prompt, so editing a task lands
    on a new key — stale generations can never be served for an edited prompt."""
    prompt = build_prompt(sit["task"])
    key = hashlib.md5(f"{prompt}|{MAX_NEW}".encode()).hexdigest()[:16]
    path = os.path.join(ORDER_CACHE, f"{sit['id']}_{key}.json")
    if os.path.exists(path) and not GEN_OVERWRITE:
        return prompt, json.load(open(path))["gen"]
    gen = generate_with_patch(mt=mt, inputs=prompt, max_new_tokens=MAX_NEW,
                              do_sample=False, remove_prefix=True)[0]
    with open(path, "w") as f:
        json.dump(dict(id=sit["id"], claim=sit["claim"], task=sit["task"], gen=gen), f, indent=1)
    return prompt, gen

todo = [s for s in SITUATIONS if ONLY is None or s["id"] in ONLY]
GENS = {}
for sit in tqdm(todo, desc="situations"):
    _, GENS[sit["id"]] = generate_for(sit)

claim = None
for sit in todo:
    if sit["claim"] != claim:
        claim = sit["claim"]
        rich_print(f"\n[bold reverse] {claim} [/bold reverse]")
    rich_print(Panel(
        Group(Text(sit["task"], style="grey70"), Text(""),
              Text.from_markup(f"[bold yellow]look for:[/bold yellow] [yellow]{escape(sit['look'])}[/yellow]"),
              Text(""),
              Panel(Text(GENS[sit["id"]].strip()), border_style="grey42", padding=(0, 1))),
        title=f"[bold]{sit['id']}[/bold]", title_align="left",
        border_style="cyan", padding=(1, 1)))

situations:   0%|          | 0/21 [00:00<?, ?it/s]

 C0 motivating 

╭─ bank ──────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are to write a python script for a bank that checks whether a given transaction is valid or not. Params     │
│ (from_account, to_account, amount). You will need to do this:                                                   │
│ * Check if the from_account has sufficient balance.                                                             │
│ * Both from and to accounts are valid from the bank database.                                                   │
│ * Check if the amount is positive.                                                                              │
│ * Assume that "bank.db" is the bank database. And, you are making sql queries with the sqlite3 library.         │
│                                                                                                                 │
│ look for: unlisted `connect` hoisted; four checks collapsed into one short-circuit `or` whose term order is     │
│ load-bearing                                                                                                    │
│                                                                                                                 │
│ ╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────╮ │
│ │ import sqlite3                                                                                              │ │
│ │                                                                                                             │ │
│ │ def is_transaction_valid(from_account, to_account, amount):                                                 │ │
│ │     conn = sqlite3.connect('bank.db')                                                                       │ │
│ │     c = conn.cursor()                                                                                       │ │
│ │                                                                                                             │ │
│ │     c.execute("SELECT balance FROM accounts WHERE account_number=?", (from_account,))                       │ │
│ │     from_balance = c.fetchone()                                                                             │ │
│ │                                                                                                             │ │
│ │     c.execute("SELECT 1 FROM accounts WHERE account_number=?", (to_account,))                               │ │
│ │     to_account_exists = c.fetchone()                                                                        │ │
│ │                                                                                                             │ │
│ │     conn.close()                                                                                            │ │
│ │                                                                                                             │ │
│ │     if not from_balance or not to_account_exists or amount <= 0 or from_balance[0] < amount:                │ │
│ │         return False                                                                                        │ │
│ │                                                                                                             │ │
│ │     return True                                                                                             │ │
│ │ </code>                                                                                                     │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
│                                                                                                                 │
╰───────────────────────────────────────────────────────

 C1 no-dep 

╭─ anti_a ────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Write a python function `check_number(n)` that returns True only if all of the following hold:                  │
│ * n is a perfect square.                                                                                        │
│ * n has more than three 1s in its binary representation.                                                        │
│ * the sum of n's digits is divisible by 3.                                                                      │
│ * n is greater than 100.                                                                                        │
│                                                                                                                 │
│ look for: 4 independent checks -> listed order                                                                  │
│                                                                                                                 │
│ ╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────╮ │
│ │ def check_number(n):                                                                                        │ │
│ │     import math                                                                                             │ │
│ │     return math.isqrt(n)**2==n and bin(n).count('1')>3 and sum(int(d) for d in str(n))%3==0 and n>100       │ │
│ │ </code>                                                                                                     │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ anti_z ────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Write a python function `check_number(n)` that returns True only if all of the following hold:                  │
│ * n is greater than 100.                                                                                        │
│ * the sum of n's digits is divisible by 3.                                                                      │
│ * n has more than three 1s in its binary representation.                                                        │
│ * n is a perfect square.                                                                                        │
│                                                                                                                 │
│ look for: same checks REVERSED -> the conjunction reverses with them                                            │
│                                                                                                                 │
│ ╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────╮ │
│ │ def check_number(n):                                                                                        │ │
│ │     return n > 100 and sum(int(d) for d in str(n)) % 3 == 0 and bin(n).count('1') > 3 and int(n**0.5)**2 == │ │
│ │ n                                                                                                           │ │
│ │ </code>                                                                                                     │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (17 outputs left out)

 C4 both 

╭─ alt_dsq ───────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are using an in-house library `dsq`. Its API contract:                                                      │
│ - A `Session` must be `.attach()`ed before any `.fetch_*()` call.                                               │
│ - After the last `.fetch_*()` call it must be `.detach()`ed.                                                    │
│                                                                                                                 │
│ Write a python function `report(sess, job_id)` that returns True only if the job succeeded, ran for under 60    │
│ seconds, and produced no warnings. You will need to do this:                                                    │
│ * Fetch the job status with `sess.fetch_status(job_id)`.                                                        │
│ * Fetch the runtime with `sess.fetch_runtime(job_id)`.                                                          │
│ * Fetch the warning count with `sess.fetch_warnings(job_id)`.                                                   │
│ * Detach the session.                                                                                           │
│ * Attach the session.                                                                                           │
│                                                                                                                 │
│ look for: ** attach() emitted TWICE: hoisted to line 1 AND again where listed **                                │
│                                                                                                                 │
│ ╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────╮ │
│ │ def report(sess, job_id):                                                                                   │ │
│ │     sess.attach()                                                                                           │ │
│ │     status = sess.fetch_status(job_id)                                                                      │ │
│ │     runtime = sess.fetch_runtime(job_id)                                                                    │ │
│ │     warnings = sess.fetch_warnings(job_id)                                                                  │ │
│ │     sess.detach()                                                                                           │ │
│ │     sess.attach()                                                                                           │ │
│ │     return status == 'succeeded' and runtime &lt; 60 and warnings == 0                                      │ │
│ │ </code>                                                                                                     │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ ctx_constrained ───────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are using an in-house library `ctxlib`. Its API contract:                                                   │
│ - A `Ctx` object must be `.prime()`d before any `.q_*()` call.                                                  │
│ - After the last `.q_*()` call it must be `.seal()`ed.                                                          │
│                                                                                                                 │
│ Write a python function `audit(ctx, user_id)` that returns True only if the role is "admin", the last login is  │
│ within 30 days, and the status is "active". You will need to do this:                                           │
│ * Query the user's role with `ctx.q_role(user_id)`.                                                             │
│ * Query the user's last login with `ctx.q_last_login(user_id)`.                                                 │
│ * Query the account status with `ctx.q_status(user_id)`.                                                        │
│ * Seal the context.                                                                                             │
│ * Prime the context.                                                                                            │
│                                                                                                                 │
│ Write one statement per step and assign each result to its own named variable.                                  │
│                                                                                                                 │
│ look for: forcing one statement per step raises duplication — the two drives are separable                      │
│                                                                                                                 │
│ ╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────╮ │
│ │ ctx.prime()                                                                                                 │ │
│ │ role = ctx.q_role(user_id)                                                                                  │ │
│ │ last_login = ctx.q_last_login(user_id)                                                                      │ │
│ │ status = ctx.q_status(user_id)                                                                              │ │
│ │ ctx.seal()                                                                                                  │ │
│ │ ctx.prime()                                                                                                 │ │
│ │ from datetime import datetime, timedelta                                                                    │ │
│ │ is_admin = role == "admin"                                                                                  │ │
│ │ login_within_30_days = (datetime.now() - last_login) &lt; timedelta(days=30)                                │ │
│ │ is_active = status == "active"                                                                              │ │
│ │ return is_admin and login_within_30_days and is_active                                                      │ │
│ │ </code>                                                                                                     │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ file_noctx ────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Write a python function `summarize_log(path)` that returns a dict with the three values below. You will need to │
│ do this:                                                                                                        │
│ * Count how many lines contain the word "ERROR".                                                                │
│ * Record the timestamp on the very first line.                                                                  │
│ * Compute the total number of bytes read.                                                                       │
│ * Close the file.                                                                                               │
│ * Open the log file at `path` for reading.                                                                      │
│                                                                                                                 │
│ Use `f = open(path)` and call `f.close()` explicitly. Do not use a `with` statement.                            │
│                                                                                                                 │
│ look for: `f = open(path)` PRODUCES a value -> no duplicate possible; cf. vault_conflict                        │
│                                                                                                                 │
│ ╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────╮ │
│ │ def summarize_log(path):                                                                                    │ │
│ │     f = open(path)                                                                                          │ │
│ │     error_count = 0                                                                                         │ │
│ │     first_timestamp = None                                                                                  │ │
│ │     total_bytes = 0                                                                                         │ │
│ │     for line in f:                                                                                          │ │
│ │         total_bytes += len(line)                                                                            │ │
│ │         if first_timestamp is None:                                                                         │ │
│ │             first_timestamp = line.split()[0]                                                               │ │
│ │         if "ERROR" in line:                                                                                 │ │
│ │             error_count += 1                                                                                │ │
│ │     f.close()                                                                                               │ │
│ │     return {'error_count': error_count, 'first_timestamp': first_timestamp, 'total_bytes': total_bytes}     │ │
│ │ </code>                                                                                                     │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯